# İSTANBUL TOPKAPI ÜNİVERSİTESİ
## DERİN ÖĞRENME ARASINAV ÖDEVİ

### Face Mask Detection - DenseNet121 & MobileNetV1
### Transfer Learning ile Yüz Maskesi Tespiti

**Öğrenci:** Ali Kemal Kefelioğlu

---

## 1. Gerekli Kütüphanelerin Yüklenmesi

- **pandas, numpy**: Veri işleme
- **sklearn (scikit-learn)**: Veri ayırma, performans metrikleri
- **TensorFlow/Keras**: Model oluşturma ve eğitimi
- **matplotlib, seaborn**: Görselleştirme

In [ ]:
# Gerekli kütüphaneleri yükle
!pip install -q kaggle

In [ ]:
import os
import shutil
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_curve, auc, roc_auc_score
)

import tensorflow as tf
from tensorflow.keras.applications import DenseNet121, MobileNet
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical

print('=' * 60)
print('TensorFlow:', tf.__version__)
print('GPU:', 'Evet' if len(tf.config.list_physical_devices('GPU')) > 0 else 'Hayır')
print('NumPy:', np.__version__)
print('Pandas:', pd.__version__)
print('=' * 60)

## 2. Veri Seti Yükleme

**Face Mask Detection Dataset** - Kaggle  
https://www.kaggle.com/datasets/omkargurav/face-mask-dataset

In [ ]:
# === KAGGLE API ANAHTARI ===
# Yöntem 1: Dosya yükleme (Colab)
from google.colab import files
print('kaggle.json dosyanızı yükleyin:')
uploaded = files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print('Kaggle API anahtarı ayarlandı.')

In [ ]:
# Veri setini indir
!kaggle datasets download -d omkargurav/face-mask-dataset --unzip -p /content/data
print('Veri seti indirildi.')

In [ ]:
# Veri seti yapısını kontrol et
!find /content/data -type d
print()

# Klasörleri bul
DATA_ROOT = '/content/data'
with_mask_dir = None
without_mask_dir = None

for root, dirs, fls in os.walk(DATA_ROOT):
    for d in dirs:
        dl = d.lower().replace(' ', '_')
        if 'with_mask' == dl or 'withmask' == dl:
            with_mask_dir = os.path.join(root, d)
        elif 'without_mask' == dl or 'withoutmask' == dl:
            without_mask_dir = os.path.join(root, d)

print(f'Maskeli: {with_mask_dir}')
print(f'Maskesiz: {without_mask_dir}')

mask_files = [f for f in os.listdir(with_mask_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))]
nomask_files = [f for f in os.listdir(without_mask_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))]
print(f'\nMaskeli görüntü: {len(mask_files)}')
print(f'Maskesiz görüntü: {len(nomask_files)}')
print(f'Toplam: {len(mask_files) + len(nomask_files)}')

## 3. Veri Setinin Train / Validation / Test Olarak Ayrılması

- Toplam verinin **%20'si test set**
- Kalan **%80** → **%80 eğitim + %20 validasyon**

Dosyaları fiziksel olarak klasörlere ayırıyoruz ki `flow_from_directory` kullanabilelim (bellek tasarrufu).

In [ ]:
# Dosya listelerini oluştur
all_files = []
all_labels = []

for f in mask_files:
    all_files.append(os.path.join(with_mask_dir, f))
    all_labels.append('with_mask')

for f in nomask_files:
    all_files.append(os.path.join(without_mask_dir, f))
    all_labels.append('without_mask')

all_files = np.array(all_files)
all_labels = np.array(all_labels)

# --- ADIM 1: %20 Test, %80 Eğitim+Validasyon ---
X_train_val, X_test, y_train_val, y_test = train_test_split(
    all_files, all_labels, test_size=0.20, random_state=42, stratify=all_labels
)

# --- ADIM 2: %80 Eğitim, %20 Validasyon ---
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.20, random_state=42, stratify=y_train_val
)

print(f'Toplam: {len(all_files)}')
print(f'Eğitim:     {len(X_train)} ({len(X_train)/len(all_files)*100:.1f}%)')
print(f'Validasyon: {len(X_val)} ({len(X_val)/len(all_files)*100:.1f}%)')
print(f'Test:       {len(X_test)} ({len(X_test)/len(all_files)*100:.1f}%)')

In [ ]:
# Dosyaları fiziksel klasörlere kopyala (flow_from_directory için)
SPLIT_DIR = '/content/split_data'

def copy_files_to_split(file_paths, labels, split_name):
    """Dosyaları train/val/test klasörlerine kopyalar."""
    for fpath, label in zip(file_paths, labels):
        dest_dir = os.path.join(SPLIT_DIR, split_name, label)
        os.makedirs(dest_dir, exist_ok=True)
        dest_path = os.path.join(dest_dir, os.path.basename(fpath))
        if not os.path.exists(dest_path):
            shutil.copy2(fpath, dest_path)

# Temizle ve yeniden oluştur
if os.path.exists(SPLIT_DIR):
    shutil.rmtree(SPLIT_DIR)

print('Dosyalar kopyalanıyor...')
copy_files_to_split(X_train, y_train, 'train')
copy_files_to_split(X_val, y_val, 'val')
copy_files_to_split(X_test, y_test, 'test')

# Doğrula
for split in ['train', 'val', 'test']:
    for cls in ['with_mask', 'without_mask']:
        path = os.path.join(SPLIT_DIR, split, cls)
        count = len(os.listdir(path)) if os.path.exists(path) else 0
        print(f'  {split}/{cls}: {count}')

print('\nKlasör yapısı hazır.')

In [ ]:
# Sınıf dağılımı grafiği
fig, ax = plt.subplots(figsize=(8, 5))
train_mask = len(os.listdir(os.path.join(SPLIT_DIR, 'train', 'with_mask')))
train_nomask = len(os.listdir(os.path.join(SPLIT_DIR, 'train', 'without_mask')))
val_mask = len(os.listdir(os.path.join(SPLIT_DIR, 'val', 'with_mask')))
val_nomask = len(os.listdir(os.path.join(SPLIT_DIR, 'val', 'without_mask')))
test_mask = len(os.listdir(os.path.join(SPLIT_DIR, 'test', 'with_mask')))
test_nomask = len(os.listdir(os.path.join(SPLIT_DIR, 'test', 'without_mask')))

data = {
    'Set': ['Eğitim', 'Eğitim', 'Validasyon', 'Validasyon', 'Test', 'Test'],
    'Sınıf': ['Maskeli', 'Maskesiz', 'Maskeli', 'Maskesiz', 'Maskeli', 'Maskesiz'],
    'Sayı': [train_mask, train_nomask, val_mask, val_nomask, test_mask, test_nomask]
}
df_dist = pd.DataFrame(data)

colors = {'Maskeli': '#2ecc71', 'Maskesiz': '#e74c3c'}
x = np.arange(3)
width = 0.35
mask_vals = [train_mask, val_mask, test_mask]
nomask_vals = [train_nomask, val_nomask, test_nomask]

bars1 = ax.bar(x - width/2, mask_vals, width, label='Maskeli', color='#2ecc71', edgecolor='black')
bars2 = ax.bar(x + width/2, nomask_vals, width, label='Maskesiz', color='#e74c3c', edgecolor='black')
ax.set_xticks(x)
ax.set_xticklabels(['Eğitim', 'Validasyon', 'Test'])
ax.set_title('Veri Seti Sınıf Dağılımı', fontsize=14, fontweight='bold')
ax.set_ylabel('Görüntü Sayısı')
ax.legend()

for bars in [bars1, bars2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 5,
                str(int(bar.get_height())), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('veri_seti_dagilimi.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Veri Arttırımı (Data Augmentation) ve Generator'lar

Eğitim verisine augmentation uygulanarak modelin genelleme yeteneği artırılır.  
`flow_from_directory` kullanarak görüntüler **batch batch** yüklenir → **RAM tasarrufu**.

In [ ]:
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

# Eğitim: augmentation + rescale
train_datagen = ImageDataGenerator(
    rescale=1.0/255,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Validasyon ve Test: sadece rescale
val_test_datagen = ImageDataGenerator(rescale=1.0/255)

# Generator'ları oluştur
train_generator = train_datagen.flow_from_directory(
    os.path.join(SPLIT_DIR, 'train'),
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

val_generator = val_test_datagen.flow_from_directory(
    os.path.join(SPLIT_DIR, 'val'),
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_generator = val_test_datagen.flow_from_directory(
    os.path.join(SPLIT_DIR, 'test'),
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print(f'\nSınıf indeksleri: {train_generator.class_indices}')
print(f'Eğitim: {train_generator.samples} görüntü')
print(f'Validasyon: {val_generator.samples} görüntü')
print(f'Test: {test_generator.samples} görüntü')

print(f'\nAugmentation parametreleri:')
print(f'  rescale=1.0/255')
print(f'  rotation_range=10')
print(f'  width_shift_range=0.1')
print(f'  height_shift_range=0.1')
print(f'  shear_range=0.1')
print(f'  zoom_range=0.1')
print(f'  horizontal_flip=True')
print(f'  fill_mode=nearest')

In [ ]:
# Örnek görüntüleri göster
fig, axes = plt.subplots(2, 5, figsize=(18, 8))
fig.suptitle('Örnek Eğitim Görüntüleri (Augmented)', fontsize=16, fontweight='bold')

batch_imgs, batch_labels = next(train_generator)
class_names = {v: k for k, v in train_generator.class_indices.items()}

for i in range(10):
    row, col = i // 5, i % 5
    axes[row, col].imshow(batch_imgs[i])
    label_idx = np.argmax(batch_labels[i])
    label_name = 'Maskeli' if class_names[label_idx] == 'with_mask' else 'Maskesiz'
    color = 'green' if label_name == 'Maskeli' else 'red'
    axes[row, col].set_title(label_name, color=color, fontweight='bold')
    axes[row, col].axis('off')

plt.tight_layout()
plt.savefig('ornek_goruntuler.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Augmentation örnekleri
fig, axes = plt.subplots(2, 5, figsize=(18, 8))
fig.suptitle('Veri Arttırımı (Data Augmentation) Örnekleri', fontsize=16, fontweight='bold')

# Augmentation'sız bir görüntü al
temp_gen = val_test_datagen.flow_from_directory(
    os.path.join(SPLIT_DIR, 'train'),
    target_size=IMAGE_SIZE, batch_size=1, class_mode=None, shuffle=True
)
original = next(temp_gen)

# Augmentation uygula
aug_gen_temp = ImageDataGenerator(
    rotation_range=10, width_shift_range=0.1, height_shift_range=0.1,
    shear_range=0.1, zoom_range=0.1, horizontal_flip=True, fill_mode='nearest'
)
aug_iter = aug_gen_temp.flow(original, batch_size=1)

axes[0, 0].imshow(original[0])
axes[0, 0].set_title('Orijinal', fontweight='bold')
axes[0, 0].axis('off')
for i in range(1, 5):
    aug = next(aug_iter)[0]
    aug = np.clip(aug, 0, 1)
    axes[0, i].imshow(aug)
    axes[0, i].set_title(f'Augmented {i}', fontweight='bold')
    axes[0, i].axis('off')

original2 = next(temp_gen)
aug_iter2 = aug_gen_temp.flow(original2, batch_size=1)
axes[1, 0].imshow(original2[0])
axes[1, 0].set_title('Orijinal', fontweight='bold')
axes[1, 0].axis('off')
for i in range(1, 5):
    aug = next(aug_iter2)[0]
    aug = np.clip(aug, 0, 1)
    axes[1, i].imshow(aug)
    axes[1, i].set_title(f'Augmented {i}', fontweight='bold')
    axes[1, i].axis('off')

plt.tight_layout()
plt.savefig('augmentation_ornekleri.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Önceden Eğitilmiş (Pre-trained) Modellerin Hazırlanması

### Model Mimarisi:
- **Base Model**: DenseNet121 / MobileNetV1 (ImageNet ağırlıkları, `include_top=False`)
- **GlobalAveragePooling2D**
- **Dropout (0.5)**: Overfitting önleme
- **Dense (256, relu)**: Tam bağlantılı katman 1
- **Dense (256, relu)**: Tam bağlantılı katman 2
- **Dense (128, relu)**: Tam bağlantılı katman 3
- **Dense (64, relu)**: Tam bağlantılı katman 4
- **Dense (2, softmax)**: Çıkış katmanı

### Eğitim Parametreleri:
- Optimizer: **Adam** (lr=0.001)
- Loss: **Categorical Crossentropy**
- **Early Stopping**: patience=25
- **ReduceLROnPlateau**: factor=0.1, patience=5
- Epoch: **100** (max)

In [ ]:
def create_model(base_model_class, model_name, image_size=(224, 224)):
    """
    Transfer learning ile model oluşturur.
    Base model dondurulur, üzerine Dense + Dropout katmanları eklenir.
    """
    print(f'\n{"="*60}')
    print(f'MODEL OLUŞTURMA: {model_name}')
    print(f'{"="*60}')

    # Base model
    base_model = base_model_class(
        weights='imagenet',
        include_top=False,
        input_shape=(image_size[0], image_size[1], 3)
    )
    base_model.trainable = False  # Dondur

    # Sınıflandırma katmanları
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.5)(x)                          # Dropout katmanı
    x = Dense(256, activation='relu')(x)          # Dense 1
    x = Dense(256, activation='relu')(x)          # Dense 2
    x = Dense(128, activation='relu')(x)          # Dense 3
    x = Dense(64, activation='relu')(x)           # Dense 4
    predictions = Dense(2, activation='softmax')(x)  # Çıkış (2 sınıf)

    model = Model(inputs=base_model.input, outputs=predictions)

    # Derleme
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    total = model.count_params()
    trainable = sum(tf.keras.backend.count_params(w) for w in model.trainable_weights)
    non_trainable = sum(tf.keras.backend.count_params(w) for w in model.non_trainable_weights)
    print(f'Toplam parametre: {total:,}')
    print(f'Eğitilebilir:     {trainable:,}')
    print(f'Eğitilemez:       {non_trainable:,}')

    return model

In [ ]:
def train_model(model, model_name, train_gen, val_gen, epochs=100):
    """
    Modeli eğitir. Early Stopping + ReduceLROnPlateau + ModelCheckpoint kullanır.
    """
    print(f'\n{"="*60}')
    print(f'MODEL EĞİTİMİ: {model_name}')
    print(f'{"="*60}')

    callbacks = [
        EarlyStopping(
            monitor='val_loss', patience=25,
            verbose=1, restore_best_weights=True, mode='min'
        ),
        ReduceLROnPlateau(
            monitor='val_loss', factor=0.1, patience=5,
            verbose=1, min_lr=1e-7, mode='min'
        ),
        ModelCheckpoint(
            filepath=f'{model_name}_best.keras',
            monitor='val_loss', save_best_only=True,
            verbose=1, mode='min'
        )
    ]

    print(f'Epoch: {epochs} | Batch: {BATCH_SIZE}')
    print(f'Optimizer: Adam (lr=0.001)')
    print(f'EarlyStopping: patience=25')
    print(f'ReduceLROnPlateau: factor=0.1, patience=5')

    history = model.fit(
        train_gen,
        epochs=epochs,
        validation_data=val_gen,
        callbacks=callbacks,
        verbose=1
    )

    print(f'\n{model_name} eğitimi tamamlandı!')
    print(f'  En iyi val_loss:     {min(history.history["val_loss"]):.4f}')
    print(f'  En iyi val_accuracy: {max(history.history["val_accuracy"]):.4f}')
    print(f'  Toplam epoch:        {len(history.history["loss"])}')

    return history

In [ ]:
def plot_training_history(history, model_name):
    """Eğitim/Validasyon Accuracy ve Loss grafiklerini çizer."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(f'{model_name} - Eğitim Grafikleri', fontsize=16, fontweight='bold')

    ep = range(1, len(history.history['loss']) + 1)

    axes[0].plot(ep, history.history['accuracy'], 'b-', label='Eğitim Accuracy', linewidth=2)
    axes[0].plot(ep, history.history['val_accuracy'], 'r-', label='Validasyon Accuracy', linewidth=2)
    axes[0].set_title('Accuracy (Doğruluk)', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
    axes[0].legend(fontsize=12); axes[0].grid(True, alpha=0.3)
    axes[0].set_ylim([0, 1.05])

    axes[1].plot(ep, history.history['loss'], 'b-', label='Eğitim Loss', linewidth=2)
    axes[1].plot(ep, history.history['val_loss'], 'r-', label='Validasyon Loss', linewidth=2)
    axes[1].set_title('Loss (Kayıp)', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
    axes[1].legend(fontsize=12); axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'{model_name}_egitim_grafikleri.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
def evaluate_model(model, model_name, test_gen):
    """
    Test seti üzerinde tahmin yapar ve tüm metrikleri hesaplar:
    ROC/AUC, Confusion Matrix, Accuracy, Precision, Recall, Specificity, F1-Score
    """
    print(f'\n{"="*60}')
    print(f'TEST DEĞERLENDİRME: {model_name}')
    print(f'{"="*60}')

    # Tahminler
    test_gen.reset()
    y_pred_proba = model.predict(test_gen, verbose=0)
    y_pred = np.argmax(y_pred_proba, axis=1)
    y_true = test_gen.classes

    # Sınıf isimleri
    class_map = {v: k for k, v in test_gen.class_indices.items()}
    # with_mask=1 (pozitif), without_mask=0 olacak şekilde düzenle
    pos_label = test_gen.class_indices.get('with_mask', 1)

    # Metrikler
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, pos_label=pos_label)
    rec = recall_score(y_true, y_pred, pos_label=pos_label)
    f1 = f1_score(y_true, y_pred, pos_label=pos_label)

    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

    auc_score = roc_auc_score(y_true, y_pred_proba[:, pos_label])

    print(f'\n📊 {model_name} - Performans Metrikleri:')
    print(f'{"─"*50}')
    print(f'  Accuracy (Doğruluk):      {acc:.4f} ({acc*100:.2f}%)')
    print(f'  Precision (Hassasiyet):   {prec:.4f} ({prec*100:.2f}%)')
    print(f'  Recall (Duyarlılık):      {rec:.4f} ({rec*100:.2f}%)')
    print(f'  Specificity (Özgüllük):   {specificity:.4f} ({specificity*100:.2f}%)')
    print(f'  F1-Score:                 {f1:.4f} ({f1*100:.2f}%)')
    print(f'  AUC:                      {auc_score:.4f}')
    print(f'{"─"*50}')
    print(f'\n📋 Confusion Matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}')

    target_names = [class_map[0], class_map[1]]
    print(f'\n📋 Sınıflandırma Raporu:')
    print(classification_report(y_true, y_pred, target_names=target_names))

    # --- Görselleştirme ---
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(f'{model_name} - Test Sonuçları', fontsize=16, fontweight='bold')

    # Confusion Matrix
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=target_names, yticklabels=target_names,
                ax=axes[0], annot_kws={'size': 16})
    axes[0].set_title('Confusion Matrix', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Tahmin', fontsize=12)
    axes[0].set_ylabel('Gerçek', fontsize=12)

    # ROC Eğrisi
    fpr, tpr, _ = roc_curve(y_true, y_pred_proba[:, pos_label])
    axes[1].plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC (AUC = {auc_score:.4f})')
    axes[1].plot([0, 1], [0, 1], 'r--', linewidth=1, label='Rastgele')
    axes[1].fill_between(fpr, tpr, alpha=0.1, color='blue')
    axes[1].set_title('ROC Eğrisi', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('False Positive Rate', fontsize=12)
    axes[1].set_ylabel('True Positive Rate', fontsize=12)
    axes[1].legend(fontsize=12)
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'{model_name}_test_sonuclari.png', dpi=150, bbox_inches='tight')
    plt.show()

    return {
        'Model': model_name,
        'Accuracy': acc, 'Precision': prec, 'Recall': rec,
        'Specificity': specificity, 'F1-Score': f1, 'AUC': auc_score,
        'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn,
        'fpr': fpr, 'tpr': tpr
    }

---
## 6. MODEL 1: DenseNet121

DenseNet121, her katmanın önceki tüm katmanlardan girdi aldığı "dense connection" mimarisini kullanır:
- Gradient vanishing problemi azalır
- Özellik yeniden kullanımı sağlanır
- Parametre verimliliği artar

In [ ]:
densenet_model = create_model(DenseNet121, 'DenseNet121', IMAGE_SIZE)
densenet_model.summary()

In [ ]:
densenet_history = train_model(densenet_model, 'DenseNet121', train_generator, val_generator, epochs=100)

In [ ]:
plot_training_history(densenet_history, 'DenseNet121')

In [ ]:
densenet_metrics = evaluate_model(densenet_model, 'DenseNet121', test_generator)

---
## 7. MODEL 2: MobileNetV1

MobileNetV1, depthwise separable convolution kullanarak:
- Hesaplama maliyetini düşürür
- Model boyutunu küçültür
- Mobil cihazlara uygun hafif model

In [ ]:
mobilenet_model = create_model(MobileNet, 'MobileNetV1', IMAGE_SIZE)
mobilenet_model.summary()

In [ ]:
# Generator'ı sıfırla (yeni epoch'tan başlasın)
train_generator_m = train_datagen.flow_from_directory(
    os.path.join(SPLIT_DIR, 'train'),
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

mobilenet_history = train_model(mobilenet_model, 'MobileNetV1', train_generator_m, val_generator, epochs=100)

In [ ]:
plot_training_history(mobilenet_history, 'MobileNetV1')

In [ ]:
mobilenet_metrics = evaluate_model(mobilenet_model, 'MobileNetV1', test_generator)

---
## 8. Model Karşılaştırması

In [ ]:
# Karşılaştırma tablosu
metrics_cols = ['Model', 'Accuracy', 'Precision', 'Recall', 'Specificity', 'F1-Score', 'AUC']
d_row = {k: densenet_metrics[k] for k in metrics_cols}
m_row = {k: mobilenet_metrics[k] for k in metrics_cols}
comparison_df = pd.DataFrame([d_row, m_row])

print('📊 Model Karşılaştırma Tablosu:')
print('=' * 80)
display(comparison_df)

In [ ]:
# Metrik karşılaştırma bar chart + ROC karşılaştırma
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('DenseNet121 vs MobileNetV1 - Karşılaştırma', fontsize=16, fontweight='bold')

metric_names = ['Accuracy', 'Precision', 'Recall', 'Specificity', 'F1-Score', 'AUC']
x = np.arange(len(metric_names))
width = 0.35

d_vals = [densenet_metrics[m] for m in metric_names]
m_vals = [mobilenet_metrics[m] for m in metric_names]

b1 = axes[0].bar(x - width/2, d_vals, width, label='DenseNet121', color='#3498db', edgecolor='black')
b2 = axes[0].bar(x + width/2, m_vals, width, label='MobileNetV1', color='#e74c3c', edgecolor='black')
axes[0].set_xticks(x)
axes[0].set_xticklabels(metric_names, rotation=45)
axes[0].set_ylim([0, 1.15])
axes[0].legend(fontsize=12)
axes[0].set_title('Performans Metrikleri', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')

for bar, val in zip(b1, d_vals):
    axes[0].text(bar.get_x()+bar.get_width()/2., bar.get_height()+0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
for bar, val in zip(b2, m_vals):
    axes[0].text(bar.get_x()+bar.get_width()/2., bar.get_height()+0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# ROC karşılaştırma
axes[1].plot(densenet_metrics['fpr'], densenet_metrics['tpr'], 'b-', linewidth=2,
             label=f'DenseNet121 (AUC={densenet_metrics["AUC"]:.4f})')
axes[1].plot(mobilenet_metrics['fpr'], mobilenet_metrics['tpr'], 'r-', linewidth=2,
             label=f'MobileNetV1 (AUC={mobilenet_metrics["AUC"]:.4f})')
axes[1].plot([0,1],[0,1],'k--', linewidth=1, label='Rastgele')
axes[1].set_title('ROC Eğrisi Karşılaştırması', fontsize=14, fontweight='bold')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend(fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('model_karsilastirmasi.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Eğitim süreçleri karşılaştırması
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Eğitim Süreçleri Karşılaştırması', fontsize=16, fontweight='bold')

axes[0].plot(densenet_history.history['accuracy'], 'b-', label='DenseNet121 Eğitim', linewidth=1.5)
axes[0].plot(densenet_history.history['val_accuracy'], 'b--', label='DenseNet121 Val', linewidth=1.5)
axes[0].plot(mobilenet_history.history['accuracy'], 'r-', label='MobileNetV1 Eğitim', linewidth=1.5)
axes[0].plot(mobilenet_history.history['val_accuracy'], 'r--', label='MobileNetV1 Val', linewidth=1.5)
axes[0].set_title('Accuracy', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(fontsize=10); axes[0].grid(True, alpha=0.3)

axes[1].plot(densenet_history.history['loss'], 'b-', label='DenseNet121 Eğitim', linewidth=1.5)
axes[1].plot(densenet_history.history['val_loss'], 'b--', label='DenseNet121 Val', linewidth=1.5)
axes[1].plot(mobilenet_history.history['loss'], 'r-', label='MobileNetV1 Eğitim', linewidth=1.5)
axes[1].plot(mobilenet_history.history['val_loss'], 'r--', label='MobileNetV1 Val', linewidth=1.5)
axes[1].set_title('Loss', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(fontsize=10); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('egitim_surecleri_karsilastirmasi.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Sonuç Raporu

In [ ]:
best = 'DenseNet121' if densenet_metrics['Accuracy'] >= mobilenet_metrics['Accuracy'] else 'MobileNetV1'

print('=' * 60)
print('SONUÇ RAPORU')
print('=' * 60)
print(f'\n🏆 En İyi Model: {best}')
print(f'\nVeri Seti: Kaggle Face Mask Detection')
print(f'Eğitim: {train_generator.samples} | Val: {val_generator.samples} | Test: {test_generator.samples}')
print(f'Görüntü: {IMAGE_SIZE[0]}x{IMAGE_SIZE[1]}x3')
print(f'\nOptimizer: Adam (lr=0.001)')
print(f'EarlyStopping: patience=25')
print(f'ReduceLROnPlateau: factor=0.1, patience=5')
print(f'Epoch: 100 (max) | Batch: {BATCH_SIZE}')
print(f'\nDenseNet121 → Acc:{densenet_metrics["Accuracy"]:.4f} F1:{densenet_metrics["F1-Score"]:.4f} AUC:{densenet_metrics["AUC"]:.4f}')
print(f'MobileNetV1 → Acc:{mobilenet_metrics["Accuracy"]:.4f} F1:{mobilenet_metrics["F1-Score"]:.4f} AUC:{mobilenet_metrics["AUC"]:.4f}')

print('\n' + '=' * 60)
display(comparison_df)

comparison_df.to_csv('model_karsilastirmasi.csv', index=False)
print('\nSonuçlar CSV olarak kaydedildi.')

---
## 10. Yorumlar

### Model Mimarisi:
- Her iki modelde de **ImageNet** ağırlıkları ile **transfer learning** uygulanmıştır.
- Base model katmanları **dondurulmuş** (freeze), sadece eklenen Dense ve Dropout katmanları eğitilmiştir.
- **Dropout (0.5)** ile overfitting önlenmiştir.
- 4 adet Dense katman (256→256→128→64) ve Softmax çıkış katmanı eklenmiştir.

### Hiperparametre Seçimleri:
- **Adam optimizer**: Adaptif öğrenme hızı sayesinde hızlı ve kararlı yakınsama sağlar.
- **ReduceLROnPlateau**: Validasyon kaybı iyileşmediğinde lr'yi 10x düşürerek ince ayar yapar.
- **Early Stopping** (patience=25): Gereksiz epoch'ları önler, en iyi ağırlıkları geri yükler.
- Başlangıç öğrenme hızı **0.001** olarak seçilmiştir.

### Veri Arttırımı:
- Eğitim verisine **rotation (10°), shift (%10), shear (%10), zoom (%10), horizontal flip** uygulanmıştır.
- Bu teknikler modelin farklı açı, pozisyon ve ölçeklerdeki görüntüleri tanımasını sağlar.
- Validasyon ve test setlerine augmentation **uygulanmamıştır** (sadece rescale).

### Model Karşılaştırması:
- **DenseNet121**: Daha derin mimari, daha fazla parametre, karmaşık özellikleri yakalamada avantajlı.
- **MobileNetV1**: Hafif mimari, hızlı eğitim ve inference, mobil uygulamalar için uygun.
- Her iki model de yüz maskesi tespitinde yüksek performans göstermiştir.

### Genel Değerlendirme:
- Transfer learning sayesinde sınırlı veri ile bile yüksek doğruluk elde edilmiştir.
- `flow_from_directory` kullanılarak bellek verimli eğitim gerçekleştirilmiştir.
- Data augmentation, modelin genelleme kapasitesini artırmıştır.
- ReduceLROnPlateau, eğitim sırasında öğrenme hızının optimum seviyeye inmesini sağlamıştır.